In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

In [3]:
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [4]:
spark

In [5]:
sc = spark.sparkContext

In [6]:
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — ready")

Spark 4.0.0-preview2 — ready


In [7]:
df = spark.read.json("cwiczenia/RTA/transactions_10k.jsonl")

print(f"Record count: {df.count()}")
df.printSchema()

Record count: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
df.show(10, truncate=False)


+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [9]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp should now be 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [10]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(avg("amount"), 2).alias("avg_PLN"),
    )
    .orderBy("store")
)

In [11]:
store_summary.show()

+--------+--------+----------+-------+
|   store|tx_count| total_PLN|avg_PLN|
+--------+--------+----------+-------+
|  Gdańsk|    2498|1021266.35| 408.83|
|  Kraków|    2522|1025896.95| 406.78|
|Warszawa|    2424| 961642.24| 396.72|
| Wrocław|    2556|1002739.21| 392.31|
+--------+--------+----------+-------+



In [12]:
from pyspark.sql.functions import min as _min, max as _max

# YOUR CODE
# df.groupBy("category").agg(...).orderBy("category").show()

In [13]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # 1-hour tumbling window
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+--------+----------+
|window                                    |tx_count|total_PLN |
+------------------------------------------+--------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150    |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661    |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189    |873403.24 |
+------------------------------------------+--------+----------+



In [14]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- total_PLN: double (nullable = true)



In [15]:
(
    hourly
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
+-------------------+-------------------+--------+----------+



In [16]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # 1h width, 30min step
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN",
    )
    .orderBy("from")
)
sliding.show(truncate=False)

+-------------------+-------------------+--------+----------+
|from               |to                 |tx_count|total_PLN |
+-------------------+-------------------+--------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112    |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150    |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443    |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661    |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696    |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189    |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749     |289709.95 |
+-------------------+-------------------+--------+----------+



In [17]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):         {tumbling_rows} windows")
print(f"Sliding  (1h / 30min): {sliding_rows} windows")

# Answer in a comment: why does sliding produce more rows?
# YOUR ANSWER:

Tumbling (1h):         3 windows
Sliding  (1h / 30min): 7 windows


In [18]:
from pyspark.sql.functions import min as _min, max as _max

category_stats = (
    df.groupBy("category")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN")
    )
    .orderBy("category")
)

category_stats.show()

+-----------+--------+----------+-------+-------+
|   category|tx_count| total_PLN|min_PLN|max_PLN|
+-----------+--------+----------+-------+-------+
|elektronika|    2542|1520770.69|    9.0| 9999.0|
|    książki|    2574| 851382.08|    5.0|9107.25|
|     odzież|    2453| 849877.55|    5.0|9696.63|
|    żywność|    2431| 789514.43|    5.0|6916.92|
+-----------+--------+----------+-------+-------+



In [19]:
windows_30_store = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "store",
        "tx_count",
        "total_PLN"
    )
    .orderBy("from", "store")
)

windows_30_store.show(truncate=False)

+-------------------+-------------------+--------+--------+---------+
|from               |to                 |store   |tx_count|total_PLN|
+-------------------+-------------------+--------+--------+---------+
|2026-04-12 08:00:00|2026-04-12 08:30:00|Gdańsk  |252     |93391.22 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Kraków  |289     |117786.42|
|2026-04-12 08:00:00|2026-04-12 08:30:00|Warszawa|275     |88441.58 |
|2026-04-12 08:00:00|2026-04-12 08:30:00|Wrocław |296     |111540.59|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Gdańsk  |514     |209187.85|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Kraków  |532     |223541.41|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Warszawa|490     |182435.06|
|2026-04-12 08:30:00|2026-04-12 09:00:00|Wrocław |502     |215587.17|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Gdańsk  |619     |253364.95|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Kraków  |590     |224358.03|
|2026-04-12 09:00:00|2026-04-12 09:30:00|Warszawa|584     |214573.66|
|2026-04-12 09:00:00

In [20]:
from pyspark.sql.functions import desc

krakow_best_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN"
    )
    .orderBy(desc("total_PLN"))
)

krakow_best_hour.show(1, truncate=False)

+-------------------+-------------------+--------+---------+
|from               |to                 |tx_count|total_PLN|
+-------------------+-------------------+--------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|1169    |483309.86|
+-------------------+-------------------+--------+---------+
only showing top 1 row



In [ ]:
# Review Questions
# 1. The 09:00–10:00 window has 4661 transactions

# 2. Difference between groupBy("store") and groupBy(window(...), "store"):
# groupBy("store") groups all transactions by store, ignoring time.
# groupBy(window(...), "store") groups transactions by both time interval and store.

# 3. In sliding window 1h / 30min, how many windows contain transactions from 09:30?
# 2 windows:
# 09:00–10:00 and 09:30–10:30.

In [21]:
gdansk_lowest_avg_hour = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(avg("amount"), 2).alias("avg_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "avg_PLN"
    )
    .orderBy("avg_PLN")
)

gdansk_lowest_avg_hour.show(1, truncate=False)

+-------------------+-------------------+--------+-------+
|from               |to                 |tx_count|avg_PLN|
+-------------------+-------------------+--------+-------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|766     |395.01 |
+-------------------+-------------------+--------+-------+
only showing top 1 row



In [22]:
from pyspark.sql.functions import hour, minute

category_9_930 = (
    df.filter(
        (hour(col("timestamp")) == 9) &
        (minute(col("timestamp")) < 30)
    )
    .groupBy("category")
    .agg(count("tx_id").alias("tx_count"))
    .orderBy("category")
)

category_9_930.show()

+-----------+--------+
|   category|tx_count|
+-----------+--------+
|elektronika|     611|
|    książki|     622|
|     odzież|     605|
|    żywność|     567|
+-----------+--------+



In [23]:
highest_15min_volume = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("tx_count"),
        _round(_sum("amount"), 2).alias("total_PLN")
    )
    .select(
        col("window.start").alias("from"),
        col("window.end").alias("to"),
        "tx_count",
        "total_PLN"
    )
    .orderBy(desc("tx_count"))
)

highest_15min_volume.show(1, truncate=False)

+-------------------+-------------------+--------+---------+
|from               |to                 |tx_count|total_PLN|
+-------------------+-------------------+--------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234    |481566.97|
+-------------------+-------------------+--------+---------+
only showing top 1 row

